# 00 – Environment Setup & Testing

Run this notebook on `ecom-dev-cluster-01` before development. It checks Spark, Unity Catalog, external locations, and ADLS access. It does not create or modify business tables.

Project values currently confirmed:
- Catalog: `adb_ecom_dev_01`
- Storage account: `stecraw001`
- Raw container: `raw`
- Curated container: `curated`


## 1. Runtime and Spark

In [0]:
from datetime import datetime, timezone
import sys

print("UTC:", datetime.now(timezone.utc).isoformat())
print("Spark:", spark.version)
print("Python:", sys.version)
print("Application ID:", spark.conf.get("spark.databricks.clusterUsageTags.applicationId", "Unavailable"))
try:
    print("Databricks runtime:", spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion"))
except Exception as e:
    print("Runtime tag unavailable:", e)


## 2. Current identity, catalog and schema

In [0]:
display(spark.sql("""
SELECT current_user() AS current_user,
       current_catalog() AS current_catalog,
       current_schema() AS current_schema
"""))

## 3. Catalog and schema visibility

In [0]:
PROJECT_CATALOG = "adb_ecom_dev_01"
display(spark.sql("SHOW CATALOGS"))
display(spark.sql(f"SHOW SCHEMAS IN `{PROJECT_CATALOG}`"))

for schema in ["bronze", "silver", "gold"]:
    try:
        spark.sql(f"DESCRIBE SCHEMA `{PROJECT_CATALOG}`.`{schema}`").show(truncate=False)
        print("PASS:", schema)
    except Exception as e:
        print("CHECK:", schema, e)


## 4. External locations and storage credentials (metadata only)

In [0]:
display(spark.sql("SHOW EXTERNAL LOCATIONS"))
display(spark.sql("SHOW STORAGE CREDENTIALS"))

for name in ["extloc_ecom_raw_dev", "extloc_ecom_curated_dev"]:
    try:
        display(spark.sql(f"DESCRIBE EXTERNAL LOCATION `{name}`"))
    except Exception as e:
        print(f"Could not describe {name}:", e)


## 5. ADLS read/list checks

In [0]:
RAW_ROOT = "abfss://raw@stecraw001.dfs.core.windows.net/"
CURATED_ROOT = "abfss://curated@stecraw001.dfs.core.windows.net/"

def list_path(path):
    try:
        items = dbutils.fs.ls(path)
        print(f"PASS: LIST {path} ({len(items)} item's)")
        for item in items[:10]:
            print(item.path, item.size)
        return True
    except Exception as e:
        print(f"FAIL: LIST {path}\n{e}")
        return False

raw_list_ok = list_path(RAW_ROOT)
curated_list_ok = list_path(CURATED_ROOT)


## 6. Optional write/read test

Keep `RUN_WRITE_TEST = False` for the first run. Enable only after confirming permissions. The test writes to a unique temporary folder under `curated/bronze/_setup_validation/`; it never touches business data. Cleanup is separately controlled.

In [0]:
from datetime import datetime, timezone

RUN_WRITE_TEST = False
CLEANUP_TEST_FOLDER = False

if RUN_WRITE_TEST:
    test_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    test_dir = f"{CURATED_ROOT}bronze/_setup_validation/{test_id}/"
    test_file = test_dir + "validation.txt"
    payload = "Azure E-commerce ETL storage validation"

    dbutils.fs.mkdirs(test_dir)
    dbutils.fs.put(test_file, payload, overwrite=False)
    actual = dbutils.fs.head(test_file, 1000)
    assert actual == payload, "Read-back did not match."
    print("PASS: write and read", test_file)

    if CLEANUP_TEST_FOLDER:
        dbutils.fs.rm(test_dir, recurse=True)
        print("Removed only this test folder:", test_dir)
    else:
        print("Retained test folder:", test_dir)
else:
    print("Write test skipped. Set RUN_WRITE_TEST = True when ready.")


## 7. Completion checklist

- [ ] Correct cluster attached and running
- [ ] Current catalog is `adb_ecom_dev_01`
- [ ] `bronze`, `silver`, `gold` schemas are visible
- [ ] Both project external locations are visible
- [ ] Raw root can be listed
- [ ] Curated root can be listed
- [ ] Optional write/read test passes

If a check fails, capture the full error and fix the relevant Unity Catalog grant, external-location scope, storage credential, or Azure RBAC/ACL. Do not bypass project storage permissions with the workspace default credential.
